# 06. Memory Optimization & Type Precision: Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **06. Memory Optimization & Type Precision**. Selecting the optimal numeric precision is the foundation of high-performance computing in NumPy. By moving from default 64-bit precision (`int64`, `float64`) to tailored subtypes (`int8`, `int16`, `int32`, `float16`, `float32`), you can slash RAM usage by 50%–87.5% and double cache hit rates. This notebook covers safe typecasting (`astype`), casting safety rules (`can_cast`, `promote_types`), integer overflow boundaries (`iinfo`), floating point machine epsilon limits (`finfo`), and type hierarchy queries (`issubdtype`).

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Typecasting with `.astype()`
- [x] 🔹 Precision Management: 50% RAM Reduction
- [x] 🔹 Integer Overflow Boundary Testing
- [x] 🔹 Inspecting Numeric Limits with `iinfo` & `finfo`


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14251 clean aligned rows):
- amounts array: shape (14251,), dtype float64
- fraud_flags array: shape (14251,), dtype int8
- account_ages array: shape (14251,), dtype float32


### 🔹 Typecasting with `.astype()`
- **What it does:** Casts numerical values from float64 (8 bytes) to float32 (4 bytes).
- **Syntax:** `.astype()`
  - **Parameters:**
    - `dtype` (*data type or dict of col -> type*): Target data type.
  - **Optional Parameters:**
    - `errors` (*{'raise', 'ignore'}, default 'raise'*): Control raising of exceptions on invalid data.
- **Key Note:** Prefer `isinstance(obj, ClassName)` over `type(obj) == ClassName` when checking types in conditional logic.
- **Dataset Application & Code Demonstration:** Applies Typecasting with `.astype()` across the extracted numeric transaction `amounts` array to compute performance metrics.


In [2]:
amounts_f32 = amounts.astype(np.float32)
print('Original Dtype:', amounts.dtype, '-> New Dtype:', amounts_f32.dtype)

Original Dtype: float64 -> New Dtype: float32


### 🔹 Precision Management: 50% RAM Reduction
- **What it does:** Measures memory savings from downcasting clean numerical values.
- **Syntax:** `function(*args, **kwargs)`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies Precision Management across the extracted numeric transaction `amounts` array to compute performance metrics.


In [3]:
print(f'float64 Memory: {amounts.nbytes / 1024:.2f} KB')
print(f'float32 Memory: {amounts_f32.nbytes / 1024:.2f} KB (exactly 50% RAM saved)')

float64 Memory: 111.34 KB
float32 Memory: 55.67 KB (exactly 50% RAM saved)


### 🔹 Integer Overflow Boundary Testing
- **What it does:** Demonstrates rollover when casting high account ages into `int8`.
- **Syntax:** `function(*args, **kwargs)`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Demonstrates Integer Overflow Boundary Testing with practical fintech data structures and variables in the following code block.


In [4]:
overflow_demo = np.array([125, 126, 127], dtype=np.int8)
overflow_demo += 1
print('int8 Two\'s Complement Overflow Result (127 + 1 -> -128):', overflow_demo)

int8 Two's Complement Overflow Result (127 + 1 -> -128): [ 126  127 -128]


### 🔹 Inspecting Numeric Limits with `iinfo` & `finfo`
- **What it does:** Inspects machine limits for `int8`, `int16`, and `float32`.
- **Syntax:** `iinfo`
  - **Optional Parameters:**
    - `memory_usage` (*bool, str, optional*): If True, include memory usage; 'deep' for deep introspection.
    - `show_counts` (*bool, optional*): Whether to show non-null counts.
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Demonstrates Inspecting Numeric Limits with `iinfo` & `finfo` with practical fintech data structures and variables in the following code block.


In [5]:
print('int8 Range:', np.iinfo(np.int8).min, 'to', np.iinfo(np.int8).max)
print('int16 Range:', np.iinfo(np.int16).min, 'to', np.iinfo(np.int16).max)
print('float32 Machine Epsilon:', np.finfo(np.float32).eps)

int8 Range: -128 to 127
int16 Range: -32768 to 32767
float32 Machine Epsilon: 1.1920929e-07


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Automated Safe Downcasting on Transaction Vectors
- **Objective:** Q1: Automated Safe Downcasting on Transaction Vectors
- **Approach:** Write a function to downcast transaction amounts and account ages to the smallest safe dtype without overflow.
- **Syntax:** `np.iinfo` checks

In [6]:
def safe_downcast(arr):
    if np.issubdtype(arr.dtype, np.floating):
        return arr.astype(np.float32)
    elif np.issubdtype(arr.dtype, np.integer):
        for dt in [np.int8, np.int16, np.int32]:
            if arr.min() >= np.iinfo(dt).min and arr.max() <= np.iinfo(dt).max:
                return arr.astype(dt)
    return arr

print('Downcasted Fraud Flags Dtype:', safe_downcast(fraud_flags).dtype)
print('Downcasted Amounts Dtype:', safe_downcast(amounts).dtype)

Downcasted Fraud Flags Dtype: int8
Downcasted Amounts Dtype: float32
